In [6]:
import cv2
import numpy as np
import mediapipe as mp
import time
import colorsys

# --- MediaPipe Setup ---
mp_holistic = mp.solutions.holistic
mp_drawing = mp.solutions.drawing_utils

# --- Fractal Settings ---
FRACTAL_WIDTH = 160  # Lower res for performance (will be upscaled)
FRACTAL_HEIGHT = 120
MAX_ITER = 20        # Lower iterations for speed
ZOOM = 1.0

# Pre-compute the coordinate grid (optimization)
y, x = np.ogrid[-1.2:1.2:FRACTAL_HEIGHT*1j, -1.8:1.8:FRACTAL_WIDTH*1j]
grid_c = x + y*1j

def media_pipe_detection(image, model):
    """Helper function to run MP detection."""
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image.flags.writeable = False
    results = model.process(image)
    image.flags.writeable = True
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    return image, results

def generate_julia_fractal(cx, cy):
    """
    Generates a Julia set image based on the complex constant c = cx + cy*i.
    Uses vectorized NumPy operations for real-time performance.
    """
    c = complex(cx, cy)
    z = grid_c.copy() * ZOOM
    
    # Initialize divergence map
    div_time = np.zeros(z.shape, dtype=int)
    
    # The Julia Iteration: z = z^2 + c
    mask = np.ones(z.shape, dtype=bool)
    for i in range(MAX_ITER):
        # Only compute for points that haven't diverged yet
        z[mask] = z[mask]**2 + c
        diverged = np.greater(np.abs(z), 2)
        div_time[mask & diverged] = i
        mask[diverged] = False
        
    # Normalize to 0-255 for visualization
    norm_fractal = (div_time / MAX_ITER * 255).astype(np.uint8)
    return norm_fractal

def get_rainbow_color(hue_offset, brightness=1.0):
    r, g, b = colorsys.hsv_to_rgb(hue_offset, 1.0, brightness)
    return (int(b * 255), int(g * 255), int(r * 255))

def get_hand_coords(landmarks, image_shape):
    h, w, c = image_shape
    coords = []
    for lm in landmarks.landmark:
        coords.append((int(lm.x * w), int(lm.y * h)))
    return coords

def draw_artistic_trail(image, history_list, trace_lifetime, current_time, base_hue_offset):
    for i, snapshot in enumerate(history_list):
        coords_curr, timestamp = snapshot
        
        age = current_time - timestamp
        life_ratio = 1.0 - (age / trace_lifetime) 
        life_ratio = max(0, min(1, life_ratio))
        
        if life_ratio <= 0: continue

        hue = (base_hue_offset + (i * 0.02)) % 1.0
        color = get_rainbow_color(hue, brightness=life_ratio)
        thickness = int(4 * life_ratio) + 1 
        radius = int(3 * life_ratio)

        # Draw connections
        for connection in mp_holistic.HAND_CONNECTIONS:
            pt1 = coords_curr[connection[0]]
            pt2 = coords_curr[connection[1]]
            cv2.line(image, pt1, pt2, color, thickness)
           
            if radius > 0:
                cv2.circle(image, pt1, radius, color, -1)
                cv2.circle(image, pt2, radius, color, -1)
        
        # Connect frames for fluid trails
        if i > 0:
            coords_prev = history_list[i-1][0]
            for j in range(21):
                pt_a = coords_prev[j]
                pt_b = coords_curr[j]
                cv2.line(image, pt_a, pt_b, color, thickness)

# --- Main Variables ---
sequence = []
threshold = 0.4
history_right = [] 
history_left = [] 
trace_lifetime = 1.5  # Shortened for snappier visual
sampling_rate = 2     # Updated for smoother trails
frame_counter = 0
global_hue = 0.0

# Current Fractal parameter (starts at 0,0)
current_cx, current_cy = -0.7, 0.27015

cap = cv2.VideoCapture(0)

with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break

        # 1. Detection
        image, results = media_pipe_detection(frame, holistic)
        h, w, c = image.shape
        
        current_time = time.time()
        frame_counter += 1
        global_hue = (global_hue + 0.01) % 1.0
        
        # 2. Update Fractal Trigger (Right Hand Index Finger)
        if results.right_hand_landmarks:
            # Index finger tip is index 8
            idx_x = results.right_hand_landmarks.landmark[8].x
            idx_y = results.right_hand_landmarks.landmark[8].y
            
            # Map screen coordinates (0 to 1) to Complex Plane (-1 to 1)
            # We smooth it slightly to prevent jitter
            target_cx = (idx_x - 0.5) * 2.0  # Range -1.0 to 1.0
            target_cy = (idx_y - 0.5) * 2.0  # Range -1.0 to 1.0
            
            # Simple lerp for smoothness
            current_cx = current_cx * 0.9 + target_cx * 0.1
            current_cy = current_cy * 0.9 + target_cy * 0.1

        # 3. Generate Fractal Overlay
        # Compute low-res fractal
        fractal_data = generate_julia_fractal(current_cx, current_cy)
        
        # Colorize (MAGMA looks like energy, TWILIGHT looks like space)
        fractal_color = cv2.applyColorMap(fractal_data, cv2.COLORMAP_TWILIGHT)
        
        # Upscale to fit screen
        fractal_overlay = cv2.resize(fractal_color, (w, h), interpolation=cv2.INTER_LINEAR)
        
        # Blend: Image * 0.7 + Fractal * 0.3
        image = cv2.addWeighted(image, 0.7, fractal_overlay, 0.5, 0)

        # 4. Standard Tracking Logic
        if results.right_hand_landmarks:
            if frame_counter % sampling_rate == 0:
                coords = get_hand_coords(results.right_hand_landmarks, image.shape)
                history_right.append((coords, current_time))
        
        if results.left_hand_landmarks:
            if frame_counter % sampling_rate == 0:
                coords = get_hand_coords(results.left_hand_landmarks, image.shape)
                history_left.append((coords, current_time))

        # Clean up old trails
        history_right = [snap for snap in history_right if (current_time - snap[1]) < trace_lifetime]
        history_left = [snap for snap in history_left if (current_time - snap[1]) < trace_lifetime]

        # Draw trails on top of the fractal
        draw_artistic_trail(image, history_right, trace_lifetime, current_time, global_hue)
        draw_artistic_trail(image, history_left, trace_lifetime, current_time, global_hue + 0.5)

        # Show feed
        cv2.imshow('Fractal Hand Tracking', image)

        if cv2.waitKey(10) & 0xFF == ord('q'):
            break
            
    cap.release()
    cv2.destroyAllWindows()

I0000 00:00:1770591268.410135   51053 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1770591268.415762   53724 gl_context.cc:369] GL version: 3.2 (OpenGL ES 3.2 Mesa 25.3.1-arch1.2), renderer: llvmpipe (LLVM 21.1.6, 256 bits)
W0000 00:00:1770591268.447257   53716 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1770591268.462420   53714 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1770591268.464233   53720 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1770591268.464397   53722 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:0